# P9 — Urban Street-Tree Health: blind exemplar

This submission uses only the provided labeled rows for feature selection and model selection. It freezes one seeded, stratified validation carve before comparing a bounded set of kNN recipes. The held-back rows appear only in the clearly marked grading register after the submitted model is final.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

SEED = 20260804

In [ ]:
try:
    trees = pd.read_csv("../data/p09_train.csv")
except FileNotFoundError:
    trees = pd.read_csv("mocktests/r1-001/data/p09_train.csv")

TARGET = "outcome"
ALL_FEATURES = [column for column in trees.columns if column != TARGET]
X = trees[ALL_FEATURES]
y = trees[TARGET]

print("shape:", trees.shape)
print("class counts:", y.value_counts().to_dict())

## Frozen validation protocol

I make one 150-row (25%) stratified carve with the single pinned seed above and never change it. Feature rankings below use only the 450-row fitting partition, so validation labels are used only for scoring candidate recipes. Absolute standardized class-mean gaps are a simple training-only filter for distance-relevant signal.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)

fit_means = X_fit.groupby(y_fit).mean()
fit_scales = X_fit.std(ddof=0).replace(0, 1)
effect_gaps = (
    (fit_means.loc["thrives"] - fit_means.loc["declines"]) / fit_scales
).abs().sort_values(ascending=False)

FEATURE_SETS = {
    "all_12": ALL_FEATURES,
    "top_10": effect_gaps.index[:10].tolist(),
    "top_7": effect_gaps.index[:7].tolist(),
}
print(effect_gaps.to_string())

## Bounded kNN campaign

The campaign compares all 12, the training-ranked top 10, and the training-ranked top 7 features; standard versus robust scaling; Manhattan versus Euclidean distance; uniform versus distance voting; and seven odd neighborhood sizes. Every candidate is a legal kNN pipeline and is evaluated on the same untouched validation carve.

In [ ]:
campaign_rows = []
for feature_set_name, feature_names in FEATURE_SETS.items():
    for scaler_name in ["standard", "robust"]:
        for p_value in [1, 2]:
            for vote_weights in ["uniform", "distance"]:
                for neighbor_count in [3, 5, 7, 9, 11, 15, 21]:
                    scaler = StandardScaler() if scaler_name == "standard" else RobustScaler()
                    candidate = Pipeline(
                        [
                            ("scaler", scaler),
                            ("knn", KNeighborsClassifier(
                                n_neighbors=neighbor_count,
                                weights=vote_weights,
                                p=p_value,
                            )),
                        ]
                    )
                    candidate.fit(X_fit[feature_names], y_fit)
                    val_predictions = candidate.predict(X_val[feature_names])
                    campaign_rows.append(
                        {
                            "feature_set": feature_set_name,
                            "scaler": scaler_name,
                            "p": p_value,
                            "weights": vote_weights,
                            "k": neighbor_count,
                            "val_f1_macro": f1_score(
                                y_val, val_predictions, average="macro"
                            ),
                        }
                    )

campaign = pd.DataFrame(campaign_rows).sort_values(
    ["val_f1_macro", "feature_set", "scaler", "p", "weights", "k"],
    ascending=[False, True, True, True, True, True],
).reset_index(drop=True)
print(campaign.head(10).to_string(index=False))

In [ ]:
accepted = campaign.iloc[0]
MODEL_FEATURES = FEATURE_SETS[accepted["feature_set"]]
validation_f1_macro = float(accepted["val_f1_macro"])

assert accepted["feature_set"] == "top_7"
assert accepted["scaler"] == "robust"
assert int(accepted["p"]) == 1
assert accepted["weights"] == "uniform"
assert int(accepted["k"]) == 5
assert np.isclose(validation_f1_macro, 0.7896213183730716, atol=1e-15, rtol=0)

baseline_model = Pipeline(
    [("scaler", StandardScaler()), ("knn", KNeighborsClassifier(n_neighbors=5))]
)
baseline_model.fit(X_fit, y_fit)
baseline_f1_macro = f1_score(
    y_val, baseline_model.predict(X_val), average="macro"
)
print("accepted features:", MODEL_FEATURES)
print(f"baseline validation f1-macro: {baseline_f1_macro:.12f}")
print(f"accepted validation f1-macro: {validation_f1_macro:.12f}")

## Final refit and prediction contract

The accepted recipe is now refit once on all 600 labeled rows. `predict_labels` never refits. The explicit empty-input branch makes “any length” include zero rows while preserving the caller's index.

In [ ]:
final_model = Pipeline(
    [
        ("scaler", RobustScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=5, weights="uniform", p=1)),
    ]
)
final_model.fit(X[MODEL_FEATURES], y)


def predict_labels(X_test):
    if len(X_test) == 0:
        return pd.Series(index=X_test.index, dtype=y.dtype, name=TARGET)
    predictions = final_model.predict(X_test[MODEL_FEATURES])
    return pd.Series(predictions, index=X_test.index, name=TARGET)

In [ ]:
for probe in [X.iloc[0:0].copy(), X.iloc[10:27].copy()]:
    probe.index = np.arange(1000, 1000 + len(probe))
    probe_predictions = predict_labels(probe)
    assert callable(predict_labels)
    assert isinstance(probe_predictions, pd.Series)
    assert len(probe_predictions) == len(probe)
    assert probe_predictions.index.equals(probe.index)
    assert set(probe_predictions.unique()) <= set(y.unique())
print("R1-R5 contract self-checks: passed for lengths 0 and 17")

In [ ]:
def deterministic_replay():
    replay_fit, replay_val, replay_y_fit, replay_y_val = train_test_split(
        X, y, test_size=150, random_state=SEED, stratify=y
    )
    replay_model = Pipeline(
        [
            ("scaler", RobustScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=5, weights="uniform", p=1)),
        ]
    )
    replay_model.fit(replay_fit[MODEL_FEATURES], replay_y_fit)
    replay_f1 = f1_score(
        replay_y_val, replay_model.predict(replay_val[MODEL_FEATURES]), average="macro"
    )
    replay_model.fit(X[MODEL_FEATURES], y)
    return replay_f1, replay_model.predict(X.iloc[200:230][MODEL_FEATURES])


(replay_f1_a, replay_pred_a), (replay_f1_b, replay_pred_b) = (
    deterministic_replay(), deterministic_replay()
)
assert replay_f1_a == replay_f1_b == validation_f1_macro
assert np.array_equal(replay_pred_a, replay_pred_b)
print("determinism audit: two exact replays matched")

## Summary (6/6 register)

**Approach.** I used a `RobustScaler` + `KNeighborsClassifier(n_neighbors=5, weights="uniform", p=1)` pipeline on seven features: canopy width, leaf chlorophyll, species diversity, soil compaction, pest damage, trunk diameter, and watering visits. Those were the seven largest absolute standardized class-mean gaps computed on the fitting partition only. I froze one stratified 150-row validation carve from 600 rows with `random_state=20260804` and used that same carve for the bounded campaign; the submitted recipe's validation macro-F1 is **0.7896213183730716**. I then refit the accepted pipeline on all 600 labeled rows.

**Intuition.** The columns have very different physical units and contain visible outliers, so robust scaling makes Manhattan neighbor distances comparable without letting a large-unit or extreme measurement dominate. Restricting distance to the strongest training-only effects also reduces dilution by weak/noise dimensions and redundant measurements. Macro-F1 fits this task because `thrives` versus `declines` is imbalanced (395 versus 205), and equal class weighting prevents strong majority-class performance from hiding poor decline detection.

**Alternatives.** The concrete all-feature baseline, `StandardScaler` + uniform Euclidean 5-NN, measured **0.7395833333333334** validation macro-F1. The campaign also tested top-10/all-12 subsets, standard scaling, Euclidean distance, distance voting, and k in {3, 5, 7, 9, 11, 15, 21}; none beat the accepted recipe on the frozen carve. Limitation: feature count and hyperparameters were selected on one validation split, so 0.7896 is selection-optimistic and may understate split-to-split uncertainty. A next step with more labeled data would be nested or repeated stratified validation, while keeping a final test set untouched.

# GRADING REGISTER — held-back evaluation, not student model selection

Everything above is the submitted artifact and was locked before this section. To reproduce the instructor-only split from the repository root, run `python mocktests/r1-001/data/gen_p09.py --with-test`; equivalently, from this notebook's directory run `python ../data/gen_p09.py --with-test`. The next cell performs those exact script semantics through IPython, then scores the already-fitted `predict_labels` once with macro-F1. The heldout result is an audit fact, never a modeling input.

In [ ]:
generator_path = "../data/gen_p09.py"
if not pd.io.common.file_exists(generator_path):
    generator_path = "mocktests/r1-001/data/gen_p09.py"
get_ipython().run_line_magic("run", generator_path + " --with-test")

In [ ]:
heldout_path = "../data/p09_heldout.csv"
if not pd.io.common.file_exists(heldout_path):
    heldout_path = "mocktests/r1-001/data/p09_heldout.csv"
heldout = pd.read_csv(heldout_path)
X_heldout = heldout[ALL_FEATURES]
y_heldout = heldout[TARGET]
heldout_predictions = predict_labels(X_heldout)
assert isinstance(heldout_predictions, pd.Series)
assert heldout_predictions.index.equals(X_heldout.index)
assert set(heldout_predictions.unique()) <= set(y.unique())
heldout_f1_macro = float(
    f1_score(y_heldout, heldout_predictions, average="macro")
)
print(f"heldout f1-macro: {heldout_f1_macro:.16f}")
print(f"heldout minus validation: {heldout_f1_macro - validation_f1_macro:.16f}")

In [ ]:
ANSWER = 0.6860096566841435
assert np.isclose(ANSWER, heldout_f1_macro, atol=1e-9, rtol=0)
ANSWER